In [ ]:
!git clone https://github.com/JoeNissen/ml-final-project

In [2]:
%cd ml-final-project

/content/ml-final-project


In [ ]:
import sys
sys.path.insert(0, "../")

!pip install rdt

from ctgan.synthesizers.ctgan import CTGANSynthesizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

from src.data_loader import load_adult_data
from src.utils import *

In [ ]:
import sys
sys.path.insert(0, "../")

!pip install rdt

from ctgan.synthesizers.ctgan import CTGANSynthesizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

from src.data_loader import load_adult_data
from src.utils import *

# Get Data

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# Load adult.csv assuming no header
D_adult = pd.read_csv("adult.csv", header=None)

# Define standard column names for the adult dataset
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
                'marital-status', 'occupation', 'relationship', 'race', 'sex',
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

# Assign these column names to the DataFrame
D_adult.columns = column_names

# Drop the first row, which contains the original string headers now incorrectly part of the data
D_adult = D_adult.iloc[1:].reset_index(drop=True)

# Convert numeric columns to appropriate types after dropping the header row
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
for col in numeric_cols:
    D_adult[col] = pd.to_numeric(D_adult[col], errors='coerce')

# Rename 'income' to 'y'
D_adult = D_adult.rename(columns={'income': 'y'})

# Extract target variable 'y' and features 'X'
y = D_adult["y"]
X = D_adult.drop(columns=["y"])

seed = 0
X_train, X_test = train_test_split(D_adult, test_size=0.6, random_state=seed)

# Train base models

In [7]:
from copy import deepcopy
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

model_dict = {
    "mlp": MLPClassifier(),
    "knn": KNeighborsClassifier(),
    "dt": DecisionTreeClassifier(),
    "rf": RandomForestClassifier(),
    "gbc": GradientBoostingClassifier(),
}

trained_model_dict = {}

# Make a copy to avoid modifying the original X_train which might be needed later
X_train_processed = X_train.copy()

# Identify categorical columns (excluding the target 'y')
categorical_cols_features = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

# Apply one-hot encoding to categorical features
X_train_processed = pd.get_dummies(X_train_processed, columns=categorical_cols_features, drop_first=True)

# Encode the target variable 'y'
le = LabelEncoder()
X_train_processed['y'] = le.fit_transform(X_train_processed['y'])

for model in model_dict.keys():
    clf = model_dict[model]
    # Use the processed data for fitting
    clf.fit(X_train_processed.drop("y", axis=1), X_train_processed["y"])

    trained_model_dict[model] = deepcopy(clf)


# Train Generative model

In [8]:
discrete_columns = [
    "workclass",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
    "y"
]

syn_model = CTGANSynthesizer(
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    generator_lr=2e-4,
    generator_decay=1e-6,
    discriminator_lr=2e-4,
    discriminator_decay=1e-6,
    batch_size=500,
    discriminator_steps=1,
    log_frequency=True,
    verbose=True,
    epochs=300,
    pac=10,
    cuda=True,
)

seed_everything(seed)
syn_model.set_random_state(seed)
syn_model.fit(train_data=X_train, discrete_columns=discrete_columns)

/usr/local/lib/python3.12/dist-packages/rdt/transformers/base.py:134: FutureWarning: Future versions of RDT will not support the 'model_missing_values' parameter. Please switch to using the 'missing_value_generation' parameter to select your strategy.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/rdt/transformers/base.py:134: FutureWarning: Future versions of RDT will not support the 'model_missing_values' parameter. Please switch to using the 'missing_value_generation' parameter to select your strategy.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/rdt/transformers/base.py:134: FutureWarning: Future versions of RDT will not support the 'model_missing_values' parameter. Please switch to using the 'missing_value_generation' parameter to select your strategy.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/rdt/transformers/base.py:134: FutureWarning: Future versions of RDT will not support the 'model_missing_values' parameter. Please switch to using the 'mi

Epoch 1, Loss G:  1.8945,Loss D: -0.5520
Epoch 2, Loss G:  1.5989,Loss D: -0.0922
Epoch 3, Loss G:  1.1338,Loss D:  0.1365
Epoch 4, Loss G:  1.1308,Loss D:  0.2748
Epoch 5, Loss G:  1.0701,Loss D: -0.0838
Epoch 6, Loss G:  0.8063,Loss D: -0.0386
Epoch 7, Loss G:  0.5061,Loss D:  0.2397
Epoch 8, Loss G:  0.4693,Loss D: -0.0859
Epoch 9, Loss G:  0.0058,Loss D:  0.0934
Epoch 10, Loss G:  0.0311,Loss D: -0.0191
Epoch 11, Loss G: -0.2129,Loss D:  0.0209
Epoch 12, Loss G: -0.7544,Loss D:  0.1306
Epoch 13, Loss G: -0.4319,Loss D: -0.1970
Epoch 14, Loss G: -0.7219,Loss D:  0.1253
Epoch 15, Loss G: -0.9348,Loss D:  0.1212
Epoch 16, Loss G: -0.8683,Loss D: -0.0116
Epoch 17, Loss G: -0.8328,Loss D:  0.0067
Epoch 18, Loss G: -0.9219,Loss D:  0.2325
Epoch 19, Loss G: -1.1538,Loss D:  0.0216
Epoch 20, Loss G: -0.9493,Loss D: -0.1305
Epoch 21, Loss G: -1.1634,Loss D:  0.1630
Epoch 22, Loss G: -1.0804,Loss D: -0.0085
Epoch 23, Loss G: -0.9369,Loss D: -0.1238
Epoch 24, Loss G: -1.2995,Loss D:  0.0433
E

# Identify column of the marginal to shift

In [9]:
from tqdm import tqdm

metric = "age"
data = X_train[metric]
cat_groups_present = False

if len(np.unique(data)) < 10:
    cat_groups = np.unique(data)
    cat_groups_present = True
else:
    mean, std = np.mean(data), np.std(data)

    minimum, maximum = np.min(data), np.max(data)

eval_idx = np.where(X_train.columns == metric)[0][0]
eval_idx


np.int64(0)

# Shift 3S

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, norm

# Redefining rejection_sample to fix the TypeError.
# This function ensures that KDE is computed only on the specified numeric feature.
def rejection_sample(D: pd.DataFrame, mean: float, std: float, feat_id: list):
    """
    Performs rejection sampling to shift the distribution of a specified numeric feature.

    Args:
        D (pd.DataFrame): The input DataFrame which may contain mixed data types.
        mean (float): The target mean for the shifted feature.
        std (float): The target standard deviation for the shifted feature.
        feat_id (list): A list containing the index of the numeric feature column to shift.

    Returns:
        np.ndarray: A numpy array of accepted samples (rows).
    """
    if not feat_id or not isinstance(feat_id, list) or len(feat_id) == 0:
        raise ValueError("feat_id must be a list containing the index of the feature column.")

    col_idx = feat_id[0]
    original_column_data = D.iloc[:, col_idx].values

    # Create KDE for the original distribution (p)
    p_orig = gaussian_kde(original_column_data.reshape(1, -1))

    # Define the target distribution (p_shifted) - a normal distribution
    p_shifted_func = lambda x: norm.pdf(x, loc=mean, scale=std)

    # Estimate M (maximum ratio) by sampling points from a reasonable range
    data_min, data_max = original_column_data.min(), original_column_data.max()
    sample_points = np.linspace(data_min - 3*std, data_max + 3*std, 1000) # Extend range to cover shifted distribution

    p_orig_at_points = p_orig(sample_points.reshape(1, -1))[0]
    p_shifted_at_points = p_shifted_func(sample_points)

    # Avoid division by zero for p_orig: if p_orig is very small, ratio can be large.
    epsilon = 1e-9 # A small value to prevent division by zero
    ratio_estimates = np.where(p_orig_at_points > epsilon, p_shifted_at_points / p_orig_at_points, 0)

    M = np.max(ratio_estimates) * 1.1 # Add a buffer to M to ensure it's an upper bound
    if M == 0: # Fallback if max ratio is 0
        M = 1.0 # Default to 1 if no ratio could be estimated, though this is a weak M

    accepted_samples_data = []
    target_num_samples = 10000 # Matching the `n=10000` from `syn_model.sample` in the original code

    if len(D) == 0:
      return np.array([])

    # Generate candidates by randomly selecting rows from D
    # Heuristic to get enough candidates, ensuring we have enough trials for rejection sampling
    num_candidates_to_generate = target_num_samples * max(int(M * 2), 10)

    candidate_indices = np.random.choice(len(D), size=num_candidates_to_generate, replace=True)

    for i in range(num_candidates_to_generate):
        if len(accepted_samples_data) >= target_num_samples:
            break

        idx = candidate_indices[i]
        candidate_row = D.iloc[idx]
        x_val = candidate_row.iloc[col_idx]

        p_s = p_shifted_func(x_val)
        p_o = p_orig(np.array([[x_val]]))[0]

        if p_o > epsilon: # Prevent division by zero
            acceptance_prob = p_s / (M * p_o)
        else:
            acceptance_prob = 0 # If original density is zero, cannot accept here

        if np.random.rand() < acceptance_prob:
            accepted_samples_data.append(candidate_row.values)

    # Convert the list of accepted rows (each as a numpy array) into a single numpy array
    return np.array(accepted_samples_data)


ys_mlp_all = []
ys_knn_all = []
ys_dt_all = []
ys_rf_all = []
ys_gbc_all = []

for i in range(2):

    ys_mlp_tmp = []
    ys_knn_tmp = []
    ys_dt_tmp = []
    ys_rf_tmp = []
    ys_gbc_tmp = []
    n_range = 10
    n_std = 1 * std

    shift_df, _ = syn_model.sample(n=10000, shift=False)

    xs = list(
        np.arange(
            mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
        )
    )
    for shift_mean in np.arange(
        mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
    ):

        reject_df = rejection_sample(
            D=shift_df, mean=shift_mean, std=std / 2, feat_id=[0]
        )
        if len(reject_df) == 0:
            continue
        test_df = pd.DataFrame(reject_df, columns=X_test.columns)
        real_tester = test_df

        # Encode the target variable 'y' and one-hot encode categorical features for the `real_tester`
        # This step is crucial because the base models were trained on processed data.
        real_tester_processed = real_tester.copy()

        # Identify categorical columns (excluding the target 'y') from X_test, as real_tester has X_test.columns
        categorical_cols_features_test = [col for col in X_test.columns if X_test[col].dtype == 'object' and col != 'y']

        # Apply one-hot encoding to categorical features that exist in the test data
        real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_test, drop_first=True)

        # Encode the target variable 'y' using the previously fitted LabelEncoder `le`
        # Ensure 'y' column is present before trying to transform
        if 'y' in real_tester_processed.columns:
            real_tester_processed['y'] = le.transform(real_tester_processed['y'])
        else:
            # Handle case where 'y' might not be in the sampled data (unlikely for this dataset structure)S
            print("Warning: 'y' column not found in real_tester_processed. Skipping label encoding for 'y'.")

        # Align columns with X_train_processed before prediction
        # Get columns from X_train_processed (excluding 'y')
        train_cols = X_train_processed.drop("y", axis=1).columns
        # Add missing columns to real_tester_processed and fill with 0
        missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
        for c in missing_cols:
            real_tester_processed[c] = 0
        # Drop extra columns from real_tester_processed
        extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
        real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))

        # Reorder columns to match the training data order
        real_tester_processed = real_tester_processed[train_cols.tolist() + ['y'] if 'y' in real_tester_processed.columns else train_cols.tolist()]

        # Ensure 'y' is dropped correctly for prediction
        X_for_prediction = real_tester_processed.drop("y", axis=1)
        y_true = real_tester_processed["y"]

        for model in model_dict.keys():
            clf = model_dict[model]
            y_pred = clf.predict(X_for_prediction)

            if model == "mlp":
                ys_mlp_tmp.append(accuracy_score(y_true, y_pred))

            if model == "knn":
                ys_knn_tmp.append(accuracy_score(y_true, y_pred))

            if model == "dt":
                ys_dt_tmp.append(accuracy_score(y_true, y_pred))

            if model == "rf":
                ys_rf_tmp.append(accuracy_score(y_true, y_pred))

            if model == "gbc":
                ys_gbc_tmp.append(accuracy_score(y_true, y_pred))

    ys_mlp_all.append(ys_mlp_tmp)
    ys_knn_all.append(ys_knn_tmp)
    ys_dt_all.append(ys_dt_tmp)
    ys_rf_all.append(ys_rf_tmp)
    ys_gbc_all.append(ys_gbc_tmp)


# Rejection sample (Test/Oracle data)

In [19]:
yr_mlp = []
yr_knn = []
yr_dt = []
yr_rf = []
yr_gbc = []
xr = list(
    np.arange(mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range)
)
i = 0
for shift_mean in np.arange(
    mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
):

    reject_df = rejection_sample(D=X_test, mean=shift_mean, std=std / 2, feat_id=[0])
    if len(reject_df) == 0:
        continue
    test_df = pd.DataFrame(reject_df, columns=X_test.columns)
    real_tester = test_df

    # Preprocessing for real_tester to match the training data format
    real_tester_processed = real_tester.copy()

    # Identify categorical columns from X_test, as real_tester has X_test.columns
    # Exclude 'y' from one-hot encoding as it will be label encoded separately
    categorical_cols_features_test = [col for col in X_test.columns if X_test[col].dtype == 'object' and col != 'y']

    # Apply one-hot encoding to categorical features that exist in the test data
    real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_test, drop_first=True)

    # Encode the target variable 'y' using the previously fitted LabelEncoder `le`
    if 'y' in real_tester_processed.columns:
        real_tester_processed['y'] = le.transform(real_tester_processed['y'])

    # Align columns with X_train_processed before prediction
    # Get columns from X_train_processed (excluding 'y')
    train_cols = X_train_processed.drop("y", axis=1).columns

    # Add missing columns to real_tester_processed and fill with 0
    missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
    for c in missing_cols:
        real_tester_processed[c] = 0

    # Drop extra columns from real_tester_processed
    extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
    real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))

    # Reorder columns to match the training data order
    # Ensure 'y' is kept separate or dropped appropriately if not used in prediction features
    real_tester_processed = real_tester_processed[train_cols.tolist() + ['y'] if 'y' in real_tester_processed.columns else train_cols.tolist()]

    X_for_prediction = real_tester_processed.drop("y", axis=1)
    y_true = real_tester_processed["y"]

    for model in model_dict.keys():
        clf = model_dict[model]
        y_score = clf.predict_proba(X_for_prediction)[:, 1]
        y_pred = clf.predict(X_for_prediction)

        if model == "mlp":
            yr_mlp.append(accuracy_score(y_true, y_pred))

        if model == "knn":
            yr_knn.append(accuracy_score(y_true, y_pred))

        if model == "dt":
            yr_dt.append(accuracy_score(y_true, y_pred))

        if model == "rf":
            yr_rf.append(accuracy_score(y_true, y_pred))

        if model == "gbc":
            yr_gbc.append(accuracy_score(y_true, y_pred))


# Shift RS (Source)

In [21]:
yr_mlp_val = []
yr_knn_val = []
yr_dt_val = []
yr_rf_val = []
yr_gbc_val = []
xr = list(
    np.arange(mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range)
)
i = 0
for shift_mean in np.arange(
    mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
):
    reject_df = rejection_sample(D=X_train, mean=shift_mean, std=std / 2, feat_id=[0])
    if len(reject_df) == 0:
        continue
    test_df = pd.DataFrame(reject_df, columns=X_train.columns)
    real_tester = test_df

    # Preprocessing for real_tester to match the training data format
    real_tester_processed = real_tester.copy()

    # Identify categorical columns from X_train (original), as real_tester has X_train.columns
    categorical_cols_features_train = [col for col in X_train.columns if X_train[col].dtype == 'object' and col != 'y']

    # Apply one-hot encoding to categorical features that exist in the test data
    real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_train, drop_first=True)

    # Encode the target variable 'y' using the previously fitted LabelEncoder `le`
    if 'y' in real_tester_processed.columns:
        real_tester_processed['y'] = le.transform(real_tester_processed['y'])

    # Align columns with X_train_processed before prediction
    # Get columns from X_train_processed (excluding 'y')
    train_cols = X_train_processed.drop("y", axis=1).columns

    # Add missing columns to real_tester_processed and fill with 0
    missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
    for c in missing_cols:
        real_tester_processed[c] = 0

    # Drop extra columns from real_tester_processed
    extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
    real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))

    # Reorder columns to match the training data order
    X_for_prediction = real_tester_processed[train_cols]
    y_true = real_tester_processed["y"]

    for model in model_dict.keys():
        clf = model_dict[model]
        y_score = clf.predict_proba(X_for_prediction)[:, 1]
        y_pred = clf.predict(X_for_prediction)

        if model == "mlp":
            yr_mlp_val.append(accuracy_score(y_true, y_pred))

        if model == "knn":
            yr_knn_val.append(accuracy_score(y_true, y_pred))

        if model == "dt":
            yr_dt_val.append(accuracy_score(y_true, y_pred))

        if model == "rf":
            yr_rf_val.append(accuracy_score(y_true, y_pred))

        if model == "gbc":
            yr_gbc_val.append(accuracy_score(y_true, y_pred))


# Mean Shift

In [22]:
yr_mlp_ms = []
yr_knn_ms = []
yr_dt_ms = []
yr_rf_ms = []
yr_gbc_ms = []
xr = list(
    np.arange(mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range)
)
i = 0
for shift_mean in np.arange(
    mean - n_std, mean + n_std, ((mean + n_std) - (mean - n_std)) / n_range
):
    from copy import deepcopy

    test_df = deepcopy(X_train)
    test_df[metric] = np.random.normal(
        loc=shift_mean, scale=std, size=len(X_train[metric])
    )

    # Removed: if len(reject_df) == 0: continue (reject_df is not defined here)

    real_tester = test_df

    # Preprocessing for real_tester to match the training data format
    real_tester_processed = real_tester.copy()

    # Identify categorical columns from X_train (original), as real_tester has X_train.columns
    categorical_cols_features_train = [col for col in X_train.columns if X_train[col].dtype == 'object' and col != 'y']

    # Apply one-hot encoding to categorical features that exist in the test data
    real_tester_processed = pd.get_dummies(real_tester_processed, columns=categorical_cols_features_train, drop_first=True)

    # Encode the target variable 'y' using the previously fitted LabelEncoder `le`
    if 'y' in real_tester_processed.columns:
        real_tester_processed['y'] = le.transform(real_tester_processed['y'])

    # Align columns with X_train_processed before prediction
    # Get columns from X_train_processed (excluding 'y')
    train_cols = X_train_processed.drop("y", axis=1).columns

    # Add missing columns to real_tester_processed and fill with 0
    missing_cols = set(train_cols) - set(real_tester_processed.drop("y", axis=1).columns)
    for c in missing_cols:
        real_tester_processed[c] = 0

    # Drop extra columns from real_tester_processed
    extra_cols = set(real_tester_processed.drop("y", axis=1).columns) - set(train_cols)
    real_tester_processed = real_tester_processed.drop(columns=list(extra_cols))

    # Reorder columns to match the training data order
    X_for_prediction = real_tester_processed[train_cols]
    y_true = real_tester_processed["y"]

    for model in model_dict.keys():
        clf = model_dict[model]
        y_score = clf.predict_proba(X_for_prediction)[:, 1]
        y_pred = clf.predict(X_for_prediction)

        if model == "mlp":
            yr_mlp_ms.append(accuracy_score(y_true, y_pred))

        if model == "knn":
            yr_knn_ms.append(accuracy_score(y_true, y_pred))

        if model == "dt":
            yr_dt_ms.append(accuracy_score(y_true, y_pred))

        if model == "rf":
            yr_rf_ms.append(accuracy_score(y_true, y_pred))

        if model == "gbc":
            yr_gbc_ms.append(accuracy_score(y_true, y_pred))


# Compare to performance on oracle/test

In [23]:
ids = np.where((X_train[metric] > xs[0]) & (X_train[metric] < xs[-1]))
quantiles = X_train[metric].iloc[ids].quantile([0.25, 0.5, 0.75]).values
q1 = xs < quantiles[0]
q2 = (xs > quantiles[0]) & (xs < quantiles[2])
q3 = xs > quantiles[2]


results = {}

q1_dict = {}
q1_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q1])
q1_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q1])
q1_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q1])

q2_dict = {}
q2_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q2])
q2_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q2])
q2_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q2])

q3_dict = {}
q3_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q3])
q3_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q3])
q3_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q3])

results["Q1"] = q1_dict
results["Q2"] = q2_dict
results["Q3"] = q3_dict


threeS_err = np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)
avg_dict = {}
avg_dict["Error 3S"] = np.mean(threeS_err)
avg_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf)))
avg_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf)))

results["avg"] = avg_dict

results


{'Q1': {'Error 3S': np.float64(0.10230000000000002),
  'Error MS': np.float64(0.1874417417417417),
  'Error RS': np.float64(0.44553333333333334)},
 'Q2': {'Error 3S': np.float64(0.1009125),
  'Error MS': np.float64(0.2438486691236691),
  'Error RS': np.float64(0.4806250000000001)},
 'Q3': {'Error 3S': np.float64(0.10873333333333333),
  'Error MS': np.float64(0.2775565656565657),
  'Error RS': np.float64(0.5034)},
 'avg': {'Error 3S': np.float64(0.103675),
  'Error MS': np.float64(0.23703895986895981),
  'Error RS': np.float64(0.47693)}}

In [24]:
ids = np.where((X_train[metric] > xs[0]) & (X_train[metric] < xs[-1]))
quantiles = X_train[metric].iloc[ids].quantile([0.25, 0.5, 0.75]).values
q1 = xs < quantiles[0]
q2 = (xs > quantiles[0]) & (xs < quantiles[2])
q3 = xs > quantiles[2]


results = {}

q1_dict = {}
q1_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q1])
q1_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q1])
q1_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q1])

q2_dict = {}
q2_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q2])
q2_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q2])
q2_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q2])

q3_dict = {}
q3_dict["Error 3S"] = np.mean(np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)[q3])
q3_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf))[q3])
q3_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf))[q3])

results["Q1"] = q1_dict
results["Q2"] = q2_dict
results["Q3"] = q3_dict


threeS_err = np.abs(np.mean(ys_rf_all, axis=0) - yr_rf)
avg_dict = {}
avg_dict["Error 3S"] = np.mean(threeS_err)
avg_dict["Error MS"] = np.mean(np.abs(np.array(yr_rf_ms) - np.array(yr_rf)))
avg_dict["Error RS"] = np.mean(np.abs(np.array(yr_rf_val) - np.array(yr_rf)))

results["avg"] = avg_dict
